# Compare the models' segmentation on Google Colab

Open this notebook from its GitHub link, so the cells are always those on `main`:
<https://colab.research.google.com/github/meiazero/garimpo-uncertainty/blob/main/notebooks/02_compare_segmentation.ipynb>

It draws `qualitative_cells` from the finished runs in Drive, on the real Sentinel-2 cells of the
unpacked store in `MyDrive/garimpo/garimpo-dataset/`. Each row is one test cell, chosen from the reference alone before
any model is read: the first empty cell, then the mining cells at the quartiles and the maximum of
reference area. The columns are its true color, its reference mask, and each model's mining
probability with the reference outlined in white, the predicted area and the intervals of the
conformal simple (CS) and COMPASS-J PCA (CJ) under it.

1. Set the form in section 1: the scale and the models to compare.
2. Run Runtime > Run all.
3. Look at the figure at the end of section 4; its PDF and SVG are in `MyDrive/garimpo/figures/`.

A CPU runtime is enough: the figure reads the runs' stored predictions, never a checkpoint. The
notebook only reads `MyDrive/garimpo/runs/`; it writes to `MyDrive/garimpo/figures/` alone.

| Section | What it does |
|---|---|
| 1. What to draw | The form |
| 2. Setup | Clones the code, builds the environment |
| 3. Drive and data | Mounts Drive, links the store and the runs in Drive, copying neither |
| 4. Figure | `garimpo figures --scale`: draws the figure, shows it, copies it to Drive |
| 5. Release | Releases the runtime |

## 1. What to draw

- `SCALE`: `pilot` for the runs on 1% of the training tiles, `full` for the experiment. `auto`
  takes the full runs once one has a checkpoint, and the pilot otherwise.
- The model boxes: the columns of the figure, in this order. A model whose run has no intervals yet
  is left out, with a message.
- `RELEASE`: release the runtime at the end.

In [ ]:
# @title The form
BRANCH = "main"  # @param {type:"string"}
SCALE = "auto"  # @param ["auto", "pilot", "full"]
UNET_SCRATCH = True  # @param {type:"boolean"}
UNET_IMAGENET = True  # @param {type:"boolean"}
UNET_SENTINEL2 = True  # @param {type:"boolean"}
CONVNET = True  # @param {type:"boolean"}
RELEASE = True  # @param {type:"boolean"}

MODELS = " ".join(
    model
    for model, chosen in (
        ("unet_scratch", UNET_SCRATCH),
        ("unet_imagenet", UNET_IMAGENET),
        ("unet_sentinel2", UNET_SENTINEL2),
        ("convnet", CONVNET),
    )
    if chosen
)

## 2. Setup

The runtime's disk is erased when a session ends, so this cell clones the code again, or moves the
clone already there to the last commit of `BRANCH`, and installs `requirements.txt`. The install
stops, and names the package, when pip would replace Colab's torch or a CUDA library, which pick
the GPU's kernels: choose the latest runtime version then. Every later cell calls a `garimpo`
command, which loads the code just installed. When a step fails, `SETUP_OK` is false and
the cells that run the code skip, so none runs an older install; the release still runs.

In [ ]:
# @title Setup: clone BRANCH and install it
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/meiazero/garimpo-uncertainty.git"
CODE = "/content/garimpo-uncertainty"  # the clone; erased with the session
steps = [] if Path(CODE, ".git").is_dir() else [["git", "clone", "-q", REPO, CODE]]
steps += [
    ["git", "-C", CODE, "fetch", "-q", "origin", BRANCH],
    ["git", "-C", CODE, "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"],
    ["git", "-C", CODE, "submodule", "update", "-q", "--init", "--recursive"],
    [sys.executable, f"{CODE}/scripts/colab_install.py"],
    ["git", "-C", CODE, "log", "-1", "--format=Running commit %h %cs %s"],
]


def ran(step: list[str]) -> bool:
    """Run one step inside the clone once it exists, as pip reads requirements.txt's -e . there."""
    try:
        done = subprocess.run(step, check=False, cwd=CODE if Path(CODE).is_dir() else None)  # noqa: S603
    except OSError as error:  # a raise here would cancel the release
        print(f"Could not start {step[0]}: {error}")
        return False
    return done.returncode == 0


# all() stops at the first failure; a later cell then skips rather than run an older install.
SETUP_OK = all(ran(step) for step in steps)
if SETUP_OK:  # a clone that failed may not be one to enter
    os.chdir(CODE)
if not SETUP_OK:
    print("The setup failed, so the cells that run the code skip: read the error above.")

## 3. Drive and data

Nothing is copied: `runs` in the clone is a link to `MyDrive/garimpo/runs/`, and
`GARIMPO_DATA_DIR` is `MyDrive/garimpo/garimpo-dataset/`. The figure reads the sidecars in `tiles/`
and, of `images.npy` and `masks.npy`, only the cells it shows; `scl.npy`, `zenodo/` and `weights/`
are never opened. It writes only to `outputs/` in the clone.
`DATA` is the unpacked store in Drive, which the figure reads in place.

In [ ]:
# @title Mount Drive, link the store and the runs
import os

from google.colab import drive

DRIVE = "/content/drive/MyDrive/garimpo"  # kept between sessions
DATA = f"{DRIVE}/garimpo-dataset"  # the unpacked store, read in place
os.environ["GARIMPO_DATA_DIR"] = DATA
try:
    drive.mount("/content/drive")
    DRIVE_OK = True
except Exception as error:  # noqa: BLE001 -- a raise here would cancel the release
    print(f"Drive did not mount, so the cells that need it skip: {error}")
    DRIVE_OK = False
if DRIVE_OK:
    !apt-get -qq install -y rsync > /dev/null
    try:  # a mount can drop after it came up, and a raise here would cancel the release
        missing = [
            name for name in ("tiles", "images.npy", "masks.npy") if not Path(DATA, name).exists()
        ]
    except OSError as error:
        print(f"Drive dropped, so the cells that need it skip: {error}")
        DRIVE_OK, missing = False, []
    for name in missing:
        print(f"Missing {DATA}/{name}: the figure cannot draw without it.")
if DRIVE_OK:
    if not os.path.lexists("runs"):  # a link to a missing folder still exists
        !ln -s {DRIVE}/runs runs
    !ls runs

## 4. Figure

`garimpo figures --scale` resolves `auto`, keeps each chosen model whose run has intervals, and
draws the figure into `outputs/compare/<scale>/`; this cell then copies that folder to
`MyDrive/garimpo/figures/<scale>/`. The audit line printed after the figure must read `clean`.
On the pilot the models trained for 40 updates, so the figure checks the drawing, not the
models.

In [ ]:
# @title Draw the segmentation comparison
import IPython.display

if not (SETUP_OK and DRIVE_OK and MODELS):
    print("Skipped: the setup failed, Drive did not mount, or no model is chosen.")
else:
    !rm -rf outputs/compare
    if Path("outputs/compare").exists():  # an old figure left there would pass for this one
        print("Could not clear outputs/compare, so nothing is drawn: read the error above.")
    else:
        !garimpo figures --scale {SCALE} --models {MODELS} --only qualitative_cells --out outputs/compare
        if globals().get("_exit_code") != 0:
            print("The draw failed, so Drive keeps its figure: read the error above.")
        elif Path("outputs/compare").is_dir():  # not drawn when no chosen model has intervals
            !mkdir -p {DRIVE}/figures
            !rsync -a outputs/compare/ {DRIVE}/figures/
            # One folder at most, named after the scale `auto` resolved to.
            for figure in Path("outputs/compare").glob("*/qualitative_cells.svg"):
                try:
                    IPython.display.display(IPython.display.SVG(filename=str(figure)))
                except Exception as error:  # noqa: BLE001 -- a raise here would cancel the release
                    print(f"Could not show {figure}: {error}")

## 5. Release

Writes the pending files to Drive and releases the runtime. No cell before this one raises a Python
error, so Run all always reaches it; the figure above stays in the notebook's output.

In [ ]:
# @title Release the runtime
import os

from google.colab import drive, runtime

if RELEASE:
    if os.path.ismount("/content/drive"):
        try:
            drive.flush_and_unmount()
        except Exception as error:  # noqa: BLE001 -- the runtime must be released whatever failed
            print(f"Drive did not confirm the last writes; check the files in Drive: {error}")
    runtime.unassign()
else:
    print("Kept: RELEASE is not checked.")